# PCAOB RAG Audit Assistant — Portfolio Walkthrough

This notebook explains the project scope, architecture, evaluation design, and reviewed results without requiring an API key or downloading a language model. The reusable implementation is located in `src/pcaob_rag/`.

## Audit problem

PCAOB inspection findings contain useful examples of audit deficiencies, but the relevant passages are distributed across long reports. This prototype tests whether retrieval-augmented generation can help a junior auditor find selected revenue-related findings, understand them in plain language, and trace each substantive claim back to a report page.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(ROOT / 'src'))

from pcaob_rag.retrieval import infer_question_scope

with (ROOT / 'config' / 'reports.json').open(encoding='utf-8') as handle:
    reports = json.load(handle)
with (ROOT / 'results' / 'evaluation_summary.json').open(encoding='utf-8') as handle:
    evaluation = json.load(handle)
with (ROOT / 'data' / 'demo_answers.json').open(encoding='utf-8') as handle:
    demo_answers = json.load(handle)

## Source scope

The prototype uses six public inspection reports: Deloitte and EY for inspection years 2022 through 2024. It isolates Part I.A and retains revenue-related text plus same-page neighboring context.

In [ ]:
report_scope = pd.DataFrame(reports)[['firm_short', 'inspection_year', 'filename']]
report_scope.sort_values(['firm_short', 'inspection_year']).reset_index(drop=True)

## Pipeline in plain English

1. **Extract:** download official PCAOB PDFs and isolate detailed Part I.A pages.
2. **Chunk and index:** create overlapping passages and represent them using TF-IDF and semantic embeddings.
3. **Retrieve:** return up to six unique cited pages; search each firm separately for cross-firm questions.
4. **Generate or refuse:** instruct Gemini to use only the retrieved text, cite substantive claims, and refuse unsupported requests.
5. **Review:** evaluate retrieval, support, citations, usefulness, and refusal behavior separately.

In [ ]:
example_question = (
    'Across the selected reports from 2022-2024, '
    'what recurring revenue-audit deficiencies appear?'
)
firms, years = infer_question_scope(example_question)
{'question': example_question, 'firms': firms, 'years': years}

## Evaluation design

The fixed benchmark contained nine answerable questions and three intentionally unsupported questions. The unsupported cases tested requests for anonymized issuer names, an undisclosed dollar amount, and an overall firm-quality ranking. Automated checks were followed by human review.

In [ ]:
metrics = evaluation['metrics']
pd.DataFrame(
    [
        ('Retrieval checks passed', f"{metrics['retrieval_checks_passed']}/{metrics['retrieval_checks_total']}"),
        ('Supported answers', f"{metrics['supported_answers']}/{metrics['answerable_questions']}"),
        ('Correct citations among supported answers', f"{metrics['correct_citations_among_supported']}/{metrics['supported_answers']}"),
        ('Correct unsupported refusals', f"{metrics['correct_refusals']}/{metrics['unsupported_questions']}"),
        ('RAG usefulness', f"{metrics['rag_usefulness_answerable']:.2f}/5"),
        ('Plain-model usefulness', f"{metrics['plain_usefulness_answerable']:.2f}/5"),
    ],
    columns=['Measure', 'Human-reviewed result'],
)

## Reviewed example

The portfolio demo displays saved outputs so a reviewer can inspect the project without an API key. The full system can optionally rebuild the corpus and run a live query using the scripts in the repository.

In [ ]:
example = next(item for item in demo_answers if item['question_id'] == 'Q05')
print('QUESTION')
print(example['question'])
print('\nGROUNDED ANSWER')
print(example['answer'])
print('\nHUMAN-REVIEW NOTE')
print(example['review_note'])

## Limitations and professional judgment

The findings apply only to this small academic benchmark. The corpus covers six reports, two firms, three years, and one audit topic. Q07 contained a citation-format issue, and Q08 over-refused despite correct retrieval. The output is coaching support—not audit evidence, authoritative guidance, or a replacement for reviewing the source and exercising professional judgment.